# Sentiment Analysis with HuggingFace `pipeline()`: A Hands-On Walkthrough

> **Focus Area:** NLP — Text Classification (Fine-Tuning Intuition & Transfer Learning)
> **Topics:** HuggingFace `pipeline()` API; Tokenizer Exploration (BPE Recap); Confidence Score Analysis; Batch Processing
> **Achievement:** `cardiffnlp/twitter-roberta-base-sentiment-latest` মডেল ব্যবহার করে একটা রিভিউ-লেভেল সেন্টিমেন্ট এক্সপ্লোরেশন নোটবুক বানানো — যেখানে লেবেল, কনফিডেন্স স্কোর, টোকেনাইজেশন, এবং ব্যাচ প্রসেসিং হাতে-কলমে দেখা যাবে।

এই নোটবুকটা `context.md`-এর কম্প্যানিয়ন — সেখানে প্রি-ট্রেইনিং বনাম ফাইন-টিউনিং, `pipeline()` API, এবং কনফিডেন্স স্কোরের থিওরি কভার করা হয়েছে। এখানে আমরা সেই একই কনসেপ্টগুলো সরাসরি কোড চালিয়ে দেখব।

---

## 1. Topic: The HuggingFace `pipeline()` API দিয়ে সেন্টিমেন্ট অ্যানালাইসিস

Class 1-এ আমরা টেক্সটকে embedding-এ রূপান্তর করা শিখেছি। এই ক্লাসে সেই embedding-ভিত্তিক বোঝাপড়া ব্যবহার করে একটা বাস্তব কাজ করব — একটা কাস্টমার রিভিউ পড়ে সেটা **Positive**, **Negative**, নাকি **Neutral** তা বলে দেওয়া, সাথে একটা কনফিডেন্স স্কোরও দেওয়া।

আমরা নিজেরা কোনো মডেল ট্রেইন করব না। বরং একটা আগে থেকে **fine-tuned** মডেল — `cardiffnlp/twitter-roberta-base-sentiment-latest` — HuggingFace-এর `pipeline()` API দিয়ে লোড করে সরাসরি ব্যবহার করব। এই নোটবুকে আমরা চারটা জিনিস হাতে-কলমে এক্সপ্লোর করব:

1. **`pipeline()` বেসিক** — মডেল লোড করে কয়েকটা এক্সাম্পল রিভিউয়ের ওপর চালানো।
2. **Tokenizer এক্সপ্লোরেশন** — টেক্সট আসলে ভেতরে ভেতরে কীভাবে টোকেনে ভাঙা হয় (Class 1-এর BPE-র সাথে কানেকশন)।
3. **Confidence Score বিশ্লেষণ** — কোন প্রেডিকশনগুলো "নিশ্চিত" আর কোনগুলো "মানুষের রিভিউয়ের জন্য ফ্ল্যাগ করা উচিত"।
4. **Batch Processing** — লুপে একটা একটা করে না পাঠিয়ে একসাথে অনেকগুলো রিভিউ পাঠানো, এবং কেন এটা এফিশিয়েন্ট।

---

## 2. Why It Is Related

প্রতিদিন হাজার হাজার কাস্টমার রিভিউ ম্যানুয়ালি পড়ে বোঝা অসম্ভব। Sentiment Analysis এই কাজটা স্বয়ংক্রিয় করে দেয় — এবং এটা ইন্ডাস্ট্রিতে সবচেয়ে বেশি ব্যবহৃত NLP অ্যাপ্লিকেশনগুলোর একটা।

`pipeline()` API-টা গুরুত্বপূর্ণ কারণ এটা আসলে দেখিয়ে দেয় প্রি-ট্রেইনড + ফাইন-টিউনড মডেল কীভাবে প্রোডাকশনে ব্যবহার করা হয় — শুধু থিওরি না, বাস্তব কোড। এই একই প্যাটার্ন (load pipeline → pass text → get label + score) NER, summarization, translation, question-answering — প্রায় সব HuggingFace টাস্কেই কাজ করে। তাই এখানে যা শিখব তা এই একটা প্রজেক্টের বাইরেও কাজে লাগবে।

---

## 3. How It Works

### 3.1 `pipeline()` ভেতরে ভেতরে কী করে

একটা `pipeline("sentiment-analysis", model=...)` কল আসলে তিনটা ধাপ একসাথে করে দেয়:

```
Raw Text
   │
   ▼
[1] Tokenizer  ──→ টেক্সটকে সাব-ওয়ার্ড টোকেনে ভাঙে, তারপর input_ids-এ কনভার্ট করে (Class 1-এর BPE)
   │
   ▼
[2] Model      ──→ টোকেনগুলো মডেলের মধ্য দিয়ে পাস হয়, প্রতিটা ক্লাসের জন্য একটা raw score (logit) বের হয়
   │
   ▼
[3] Softmax    ──→ logits-কে প্রবাবিলিটিতে রূপান্তর করে (সব ক্লাস মিলিয়ে যোগফল = ১)
   │
   ▼
Label + Confidence Score  (যেমন: {'label': 'negative', 'score': 0.94})
```

`pipeline()` এই পুরো জিনিসটা এক লাইনে wrap করে দেয় — আমাদের আলাদা করে tokenizer, model, softmax কিছুই ম্যানুয়ালি চালাতে হয় না। কিন্তু নিচে আমরা tokenizer অংশটা আলাদাভাবেও চালিয়ে দেখব, যাতে ভেতরের কাজটা বোঝা যায়।

### 3.2 Confidence Score-এর মানে

মডেল শুধু একটা লেবেল দেয় না — একটা স্কোরও দেয় (০ থেকে ১)। এই স্কোর মডেলের "নিশ্চয়তা" বোঝায়। ০.৯৪ মানে মডেল প্রায় নিশ্চিত; ০.৫২ মানে মডেল দ্বিধায় আছে। প্রোডাকশন সিস্টেমে সাধারণত একটা থ্রেশহোল্ড (যেমন ৬০%) ঠিক করে দেওয়া হয় — তার নিচে স্কোর এলে সেটা "Needs Human Review" হিসেবে ফ্ল্যাগ হয়।

### 3.3 Batch Processing

`pipeline()`-কে একটা একটা করে স্ট্রিং না পাঠিয়ে একটা লিস্ট পাঠালে এটা ব্যাচে প্রসেস করে — মডেলের ভেতরে ম্যাট্রিক্স অপারেশনগুলো একসাথে সব রিভিউয়ের ওপর চলে, প্রতিটার জন্য আলাদা ফাংশন কল লাগে না। এটা ঠিক Week 2-র **Vectorization** লেকচারের একই আইডিয়া — Python-এর `for` লুপে এক এক করে প্রসেস করার বদলে, আন্ডারলাইং লাইব্রেরিকে (এখানে PyTorch/TensorFlow) একবারে পুরো ব্যাচ দিয়ে দেওয়া, যাতে সে অপ্টিমাইজড, প্যারালালাইজড কোড দিয়ে সবগুলো একসাথে হ্যান্ডল করতে পারে।

---

## 4. Details & Valid Points

### 4.1 Model Reference Table — কোন সেন্টিমেন্ট মডেল কখন ব্যবহার করবেন

| Model | Use Case | Advantage | Limitation |
| --- | --- | --- | --- |
| **`cardiffnlp/twitter-roberta-base-sentiment-latest`** (এই নোটবুকে ব্যবহৃত) | ৩-ক্লাস (Pos/Neg/Neutral) জেনারেল সেন্টিমেন্ট | সত্যিকারের Neutral ক্লাস আছে, কোনো hack লাগে না | টুইটার-স্টাইল টেক্সটে ট্রেইনড; প্রোডাক্ট রিভিউতে সামান্য ডোমেইন-শিফট থাকতে পারে |
| `distilbert-base-uncased-finetuned-sst-2-english` | দ্রুত বাইনারি (শুধু Pos/Neg) সেন্টিমেন্ট | ছোট, ফাস্ট, খুবই জনপ্রিয় ডিফল্ট | Neutral ক্লাস নেই — জোর করে থ্রেশহোল্ড বসাতে হয়, যা অনির্ভরযোগ্য |
| `nlptown/bert-base-multilingual-uncased-sentiment` | ১-৫ স্টার রেটিং প্রেডিকশন | মাল্টিলিঙ্গুয়াল, রেটিং-স্টাইল আউটপুট | Positive/Negative/Neutral-এর বদলে স্টার-স্কেল, ম্যাপিং লাগবে |
| OpenAI GPT (zero-shot prompt দিয়ে) | কাস্টম ক্যাটাগরি সহ ফ্লেক্সিবল ক্লাসিফিকেশন | কোনো ফাইন-টিউনিং লাগে না, প্রম্পট বদলালেই যথেষ্ট | লোকাল না, প্রতি কলে খরচ, ডেটা থার্ড-পার্টিতে যায় |

> **Valid Point:** আমরা `cardiffnlp/twitter-roberta-base-sentiment-latest` বেছে নিয়েছি কারণ এই প্রজেক্টে আমাদের সত্যিকারের ৩-ক্লাস আউটপুট (Positive/Negative/Neutral) দরকার — শুধু "সবচেয়ে জনপ্রিয়" মডেল (DistilBERT SST-2) বেছে নিলে Neutral ক্লাস পাওয়া যেত না।

### 4.2 Domain Shift — একটা সৎ সতর্কতা

এই মডেল টুইটের ওপর ট্রেইনড, প্রোডাক্ট রিভিউয়ের ওপর না। ফলে প্রোডাক্ট রিভিউয়ের মতো ফরমাল টেক্সটে মাঝে মাঝে কনফিডেন্স একটু কম হতে পারে — কিন্তু সাধারণ Positive/Negative প্যাটার্ন ডোমেইন জুড়ে যথেষ্ট মিল থাকে বলে এটি এখনও কার্যকর।

---

---

## 5. Achievement: হাতে-কলমে `pipeline()` এক্সপ্লোরেশন

নিচে আমরা ধাপে ধাপে কোড চালিয়ে ওপরের প্রতিটা কনসেপ্ট বাস্তবে দেখব।

> **Note:** এই এনভায়রনমেন্টে গ্যারান্টিড ইন্টারনেট/প্যাকেজ অ্যাক্সেস নাও থাকতে পারে, তাই এই নোটবুকের কোড সেল actually execute করা হয়নি — কিন্তু কোড সঠিক এবং একটা ইন্টারনেট-কানেক্টেড, `transformers`+`torch` ইনস্টল করা এনভায়রনমেন্টে রান করলে কাজ করবে।

প্রথমে দরকারি লাইব্রেরি ইনস্টল ও ইম্পোর্ট করা যাক।

In [ ]:
# প্রয়োজনীয় লাইব্রেরি ইনস্টল (Colab/fresh environment হলে uncomment করুন)
# !pip install -q transformers torch pandas

from transformers import pipeline, AutoTokenizer
import pandas as pd

MODEL_NAME = "cardiffnlp/twitter-roberta-base-sentiment-latest"
print(f"Using model: {MODEL_NAME}")

### 5.1 `pipeline()` বেসিক — কয়েকটা এক্সাম্পল রিভিউ চালানো

এখন আমরা `pipeline("sentiment-analysis", ...)` লোড করব এবং চারটা ভিন্ন ধরনের রিভিউ দিয়ে টেস্ট করব:

* একটা স্পষ্টভাবে **Positive** রিভিউ
* একটা স্পষ্টভাবে **Negative** রিভিউ
* একটা সত্যিকারের **Neutral/Mixed** রিভিউ (কিছু ভালো, কিছু খারাপ)
* একটা **Sarcastic** রিভিউ (Brain Teaser #1-এর সাথে সম্পর্কিত — শব্দে Positive, অর্থে Negative)

In [ ]:
# pipeline() দিয়ে মডেল লোড করা — tokenizer, model, ও post-processing সব একসাথে
classifier = pipeline("sentiment-analysis", model=MODEL_NAME)

example_reviews = [
    "Absolutely love this product, best purchase I've made all year!",   # clearly positive
    "The product broke after two days, very disappointed.",              # clearly negative
    "It arrived on time. The box was blue. It works as described.",      # genuinely neutral
    "Oh great, ANOTHER broken product. Just what I needed.",             # sarcastic
]

results = classifier(example_reviews)

for review, result in zip(example_reviews, results):
    print(f"Review:     {review}")
    print(f"Label:      {result['label']}")
    print(f"Confidence: {result['score']:.4f}")
    print("-" * 60)

**যা লক্ষ্য করার বিষয়:** প্রথম দুটো রিভিউতে মডেল খুব উচ্চ কনফিডেন্সে সঠিক লেবেল দেবে, কারণ ভাষাটা স্পষ্ট। তৃতীয়টা (neutral) হয়তো তুলনামূলক কম কনফিডেন্সে আসবে, কারণ কোনো strong sentiment শব্দ নেই। চতুর্থটা (sarcasm) — এটাই সবচেয়ে ইন্টারেস্টিং কেস: মডেল "great" শব্দটা দেখে Positive-এর দিকে ঝুঁকে যেতে পারে, যদিও বাক্যটার আসল অর্থ Negative। এটাই দেখায় মডেল শব্দের প্যাটার্ন চেনে, কিন্তু মানুষের মতো কনটেক্সট বা টোন পুরোপুরি বোঝে না — sarcasm ধরা এখনও একটা open challenge।

---

### 5.2 Tokenizer এক্সপ্লোরেশন — ভেতরে ভেতরে কী হচ্ছে

`pipeline()` ভেতরে ভেতরে একটা tokenizer ব্যবহার করে টেক্সটকে সংখ্যায় রূপান্তর করে। Class 1-এ আমরা **BPE (Byte-Pair Encoding)** শিখেছিলাম — কীভাবে শব্দকে সাব-ওয়ার্ড টুকরায় ভাঙা হয়। এখন সেই একই মডেলের tokenizer সরাসরি লোড করে দেখব সেটা বাস্তবে কী করে।

In [ ]:
# একই মডেলের tokenizer সরাসরি লোড করা
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

sample_sentence = "The product broke after two days, very disappointed."

# ধাপ ১: টেক্সটকে সাব-ওয়ার্ড টোকেনে ভাঙা (BPE-স্টাইল)
tokens = tokenizer.tokenize(sample_sentence)
print("Tokens:")
print(tokens)
print()

# ধাপ ২: টোকেনগুলোকে input_ids + attention_mask-এ কনভার্ট করা
encoded = tokenizer(sample_sentence)
print("Input IDs:")
print(encoded["input_ids"])
print()
print("Attention Mask:")
print(encoded["attention_mask"])
print()
print(f"মোট টোকেন সংখ্যা (special tokens সহ): {len(encoded['input_ids'])}")

**Class 1-এর সাথে কানেকশন:** লক্ষ্য করুন `tokenize()`-এর আউটপুটে কিছু শব্দ (যেমন "disappointed") হয়তো একটা সিঙ্গেল টোকেন থাকবে, আবার কিছু কম-কমন শব্দ একাধিক সাব-ওয়ার্ড টুকরায় ভেঙে যেতে পারে (যেমন `Ġdisappoint` + `ed`) — এটাই BPE-র মূল আইডিয়া: পুরো ভোকাবুলারি মুখস্থ না করে, ঘন ঘন ব্যবহৃত সাব-ওয়ার্ড ইউনিট দিয়ে যেকোনো শব্দ তৈরি করা যায়, এমনকি নতুন/আনকমন শব্দও। `input_ids` হলো সেই টোকেনগুলোর ভোকাবুলারি-ইনডেক্স, আর `attention_mask` বলে দেয় কোন পজিশনগুলো আসল টোকেন (১) আর কোনগুলো padding (০) — ব্যাচে একাধিক বাক্যের দৈর্ঘ্য আলাদা হলে এটা দরকার হয়।

---

### 5.3 Confidence Score বিশ্লেষণ — কখন মানুষের রিভিউ দরকার

এখন পর্যন্ত আমরা শুধু "সবচেয়ে বেশি স্কোরের" ক্লাসটা দেখেছি। কিন্তু আসলে মডেল প্রতিটা ক্লাসের (Positive/Negative/Neutral) জন্যই একটা স্কোর বের করে — `top_k=None` (নতুন `transformers` ভার্সনে; পুরনো ভার্সনে `return_all_scores=True`) দিলে আমরা সবগুলো ক্লাসের স্কোর একসাথে দেখতে পারি।

এই সেকশনে আমরা:
1. কয়েকটা রিভিউর ওপর সব-ক্লাস স্কোর বের করব।
2. একটা `pandas` DataFrame বানাব।
3. একটা কনফিডেন্স থ্রেশহোল্ড (৬০%) ঠিক করব — এর নিচে top score এলে সেই রিভিউকে **"Needs Human Review"** হিসেবে ফ্ল্যাগ করব (context.md-এর Brain Teaser #2)।

In [ ]:
# সব ক্লাসের স্কোর পাওয়ার জন্য top_k=None দিয়ে pipeline লোড করা
classifier_all_scores = pipeline("sentiment-analysis", model=MODEL_NAME, top_k=None)
# পুরনো transformers ভার্সনে সমতুল্য: pipeline(..., return_all_scores=True)

batch_reviews = [
    "Absolutely love this product, best purchase I've made all year!",
    "The product broke after two days, very disappointed.",
    "It arrived on time. The box was blue. It works as described.",
    "Oh great, ANOTHER broken product. Just what I needed.",
    "Decent for the price, but shipping took forever.",
    "Customer support was unhelpful and rude.",
    "Works exactly as advertised, no complaints.",
    "I'm not sure how I feel about this one.",
]

all_scores = classifier_all_scores(batch_reviews)

CONFIDENCE_THRESHOLD = 0.60

rows = []
for review, score_list in zip(batch_reviews, all_scores):
    # score_list হলো [{'label': 'positive', 'score': 0.8}, {'label': 'negative', ...}, ...]
    score_list_sorted = sorted(score_list, key=lambda d: d["score"], reverse=True)
    top = score_list_sorted[0]

    row = {"review": review, "predicted_label": top["label"], "top_score": top["score"]}
    for entry in score_list:
        row[f"score_{entry['label']}"] = entry["score"]
    row["needs_human_review"] = top["score"] < CONFIDENCE_THRESHOLD

    rows.append(row)

df = pd.DataFrame(rows)
df

In [ ]:
# শুধু ফ্ল্যাগড (কম-কনফিডেন্স) রিভিউগুলো দেখা
flagged = df[df["needs_human_review"]]
print(f"থ্রেশহোল্ড {CONFIDENCE_THRESHOLD:.0%}-এর নিচে থাকা রিভিউ সংখ্যা: {len(flagged)} / {len(df)}")
flagged[["review", "predicted_label", "top_score"]]

**থ্রেশহোল্ড নিয়ে ভাবনা (Brain Teaser #2):** ৬০% একটা মাঝামাঝি পছন্দ। থ্রেশহোল্ড খুব বেশি রাখলে (যেমন ৯০%) — অনেক আসলে-ঠিক প্রেডিকশনও অযথা "human review"-তে চলে যাবে, যা রিভিউ টিমের কাজ বাড়িয়ে দেবে (false positives বেশি)। থ্রেশহোল্ড খুব কম রাখলে (যেমন ৩০%) — সত্যিকারের অস্পষ্ট/ভুল প্রেডিকশনগুলোও ফ্ল্যাগ ছাড়াই পাস হয়ে যাবে, যা প্রোডাকশনে সাইলেন্ট ভুলের ঝুঁকি বাড়ায় (false negatives বেশি)। সঠিক থ্রেশহোল্ড নির্ভর করে — ভুল প্রেডিকশনের কস্ট কত বেশি, বনাম মানুষের রিভিউ করার ক্যাপাসিটি কতটুকু আছে, তার ওপর।

---

### 5.4 Batch Processing — লুপ না, একটা কল

নিচে ১০টা রিভিউ একসাথে একটা লিস্ট হিসেবে `classifier`-কে পাঠানো হচ্ছে — `for` লুপে একটা একটা করে না পাঠিয়ে। এটা ঠিক Week 2-র Vectorization লেকচারের সেই একই নীতি: **"process many things in one call, not a Python for-loop।"** নিচের দুটো অ্যাপ্রোচ তুলনা করা হলো (কোড হিসেবে দুটোই লেখা আছে, কিন্তু বাস্তবে ব্যাচ ভার্সনটাই ব্যবহার করা উচিত)।

> **Performance note (conceptual, actual runtime নয়):** লুপ ভার্সনে প্রতিটা রিভিউর জন্য আলাদা করে মডেল-কল হয় — প্রতিবার টোকেনাইজেশন + ফরওয়ার্ড পাস আলাদা আলাদাভাবে সেটআপ হয়, যা ওভারহেড বাড়ায়। ব্যাচ ভার্সনে একবারেই সবগুলো রিভিউর টোকেনগুলো একটা ম্যাট্রিক্সে সাজিয়ে, একটাই ফরওয়ার্ড পাসে সবগুলোর রেজাল্ট বের করা হয় — GPU/CPU-র প্যারালাল কম্পিউট ক্ষমতা পুরোপুরি কাজে লাগে, ফলে ১০টা রিভিউর জন্য ১০ বার কল করার চেয়ে অনেক দ্রুত হয়।

In [ ]:
ten_reviews = [
    "Absolutely love this product, best purchase I've made all year!",
    "The product broke after two days, very disappointed.",
    "It arrived on time. The box was blue. It works as described.",
    "Oh great, ANOTHER broken product. Just what I needed.",
    "Decent for the price, but shipping took forever.",
    "Customer support was unhelpful and rude.",
    "Works exactly as advertised, no complaints.",
    "I'm not sure how I feel about this one.",
    "Five stars, would buy again in a heartbeat!",
    "Waste of money, do not recommend to anyone.",
]

# --- Anti-pattern: এক এক করে লুপে কল করা (ধীর, non-vectorized) ---
# results_loop = []
# for review in ten_reviews:
#     results_loop.append(classifier(review))

# --- Correct pattern: একবারে পুরো ব্যাচ পাঠানো (fast, vectorized) ---
results_batch = classifier(ten_reviews)

for review, result in zip(ten_reviews, results_batch):
    print(f"[{result['label']:>8}] ({result['score']:.3f})  {review}")

---

## 6. Summary

এই নোটবুকে আমরা `cardiffnlp/twitter-roberta-base-sentiment-latest` মডেল দিয়ে চারটা জিনিস হাতে-কলমে দেখলাম:

1. **`pipeline()` বেসিক** — এক লাইনে মডেল লোড করে Positive/Negative/Neutral/Sarcastic রিভিউর ওপর প্রেডিকশন চালানো।
2. **Tokenizer এক্সপ্লোরেশন** — `AutoTokenizer`-এর মাধ্যমে টেক্সট সাব-ওয়ার্ড টোকেনে ভাঙা এবং `input_ids`/`attention_mask` দেখা (Class 1-এর BPE-র বাস্তব প্রয়োগ)।
3. **Confidence Score বিশ্লেষণ** — `top_k=None` দিয়ে সব-ক্লাস স্কোর বের করে একটা DataFrame বানানো, এবং থ্রেশহোল্ড দিয়ে কম-কনফিডেন্স রিভিউ ফ্ল্যাগ করা।
4. **Batch Processing** — লুপের বদলে একবারে লিস্ট পাঠিয়ে ভেক্টরাইজড ইনফারেন্স করা (Week 2-র Vectorization নীতির পুনরাবৃত্তি)।

এই একই প্যাটার্ন — `pipeline()` লোড করা, টেক্সট/ব্যাচ পাঠানো, স্কোর ইন্টারপ্রেট করা — প্রায় সব HuggingFace NLP টাস্কেই কাজ করে, শুধু `pipeline()`-এর প্রথম আর্গুমেন্ট (task name) আর মডেল বদলে যায়।

---

## 🧠 Brain Teasers & Exercises (নিজে রান করে দেখুন)

1. **Sarcasm Test**: উপরে `example_reviews`-এ থাকা sarcastic বাক্যটা ("Oh great, ANOTHER broken product...") মডেল আসলে কী লেবেল দিয়েছে দেখুন। এটা কি সঠিকভাবে Negative ধরতে পেরেছে, নাকি "great" শব্দে বিভ্রান্ত হয়ে Positive/Neutral দিয়েছে? নিজে আরও ২-৩টা sarcastic বাক্য বানিয়ে টেস্ট করুন — প্যাটার্নটা কি সামঞ্জস্যপূর্ণ?

2. **Confidence Threshold**: Section 5.3-এর `CONFIDENCE_THRESHOLD = 0.60` বদলে ৯০% এবং ৩০% দিয়ে আবার চালান। কতগুলো রিভিউ ফ্ল্যাগ হয় প্রতিবার? খুব বেশি থ্রেশহোল্ডে false positive (ঠিক প্রেডিকশনও ফ্ল্যাগ হওয়া) আর খুব কম থ্রেশহোল্ডে false negative (ভুল প্রেডিকশন ফ্ল্যাগ না হওয়া) — এই ট্রেড-অফটা ডেটাতে নিজে দেখুন।

3. **Domain Shift Experiment**: একটা টুইটার-স্টাইল বাক্য ("ugh this app is trash 🗑️") আর একটা ফরমাল রিভিউ বাক্য ("The application crashes frequently and lacks essential features.") — দুটোই কাছাকাছি অর্থ বহন করে। `classifier_all_scores` দিয়ে দুটো চালিয়ে দেখুন কোনটায় top score বেশি। মডেল টুইটার-স্টাইল টেক্সটে বেশি কনফিডেন্ট হয় কিনা, এবং কেন (মনে করুন — মডেল কোন ডেটাতে ট্রেইনড হয়েছিল)?